# Lesson 7A: Deep Q-Networks (DQN) - Theory

## Introduction

Previous lessons scaled tabular RL with hand-crafted features (tile coding, RBF). Now: **end-to-end learning** with neural networks.

Challenge: Q-learning with neural networks diverges (deadly triad). DQN solves this with two key ideas:
1. **Experience replay**: Break temporal correlation in data
2. **Fixed Q-targets**: Use an old network copy as the target

Result: Stable, scalable, deep learning RL.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

print('Deep Q-Networks (DQN)')

## The Problem: Neural Q-Learning Diverges

### Tabular Q-Learning (Stable)
$$Q(s,a) \leftarrow Q(s,a) + \alpha [R + \gamma \max_a Q(s',a) - Q(s,a)]$$

Converges under reasonable conditions (finite state space, decaying α).

### Neural Q-Learning (Unstable)
Replace Q-table with network: $Q(s,a;\theta)$
$$L(\theta) = (R + \gamma \max_a Q(s',a;\theta) - Q(s,a;\theta))^2$$

**Problems**:
1. **Correlated data**: Sequential samples from environment are correlated. SGD assumes i.i.d.
2. **Non-stationary targets**: Target uses current $\theta$, but $\theta$ changes → moving target
3. **Deadly triad**: Approximation + bootstrapping + off-policy → divergence

## Solution 1: Experience Replay

### Idea
Store transitions $(s, a, r, s')$ in a **replay buffer**. Sample random minibatches to break correlation.

### Algorithm
```
1. Initialize replay buffer D with capacity N
2. Loop:
   a. Execute action via ε-greedy, observe (s, a, r, s')
   b. Store in D: D ← D ∪ {(s, a, r, s')}
   c. Sample minibatch from D: {(si, ai, ri, s'i)}
   d. Update Q-network on minibatch
```

### Effect
- Data is no longer i.i.d. in time, but random samples are i.i.d. within minibatch
- Allows reusing old experiences (off-policy)
- Decouples data collection from learning

### Stability Gain
Reduces correlation → SGD works better → convergence more likely

## Solution 2: Fixed Q-Targets

### The Non-Stationary Target Problem

Loss depends on target which depends on θ:
$$L(\theta) = (R + \gamma \max_a Q(s',a;\theta) - Q(s,a;\theta))^2$$

When we update θ, the target changes → moving target → hard to chase.

### Solution: Target Network

Keep a **separate copy** of Q-network (θ⁻) for computing targets:
$$L(\theta) = (R + \gamma \max_a Q(s',a;\theta^-) - Q(s,a;\theta))^2$$

Update θ with gradient descent, but **freeze θ⁻ for N steps**, then copy θ → θ⁻.

### Effect
- Target is now stable for N steps (constant)
- Network chases a moving target, but slowly
- Significantly improves stability

## DQN Algorithm

### Pseudocode
```
Initialize:
  Q-network with weights θ
  Target network θ⁻ = θ
  Replay buffer D
  
Loop over episodes:
  Initialize state s
  
  Loop over steps:
    1. ε-greedy: a = argmax_a Q(s;θ) with probability 1-ε, random otherwise
    2. Execute: (r, s') ← env.step(a)
    3. Store: D.append((s, a, r, s'))
    
    4. Sample minibatch from D
    5. For each transition in minibatch:
       y = r + γ * max_a Q(s', a; θ⁻)  [target]
       L = (y - Q(s, a; θ))²
    6. Gradient descent: θ ← θ - α ∇_θ L
    
    7. Every N_target steps: θ⁻ ← θ  [update target]
    
    s ← s'
```

## Key Design Choices

### Epsilon Decay
- Start high (ε=1.0): explore uniformly
- Decay over time: ε → ε_min (e.g., 0.01)
- Balances exploration vs exploitation

### Replay Buffer Size
- Larger → more diversity, but more memory
- Typical: 10k to 1M transitions
- Trade-off: old data (stale) vs storage

### Target Network Update Frequency
- N_target = 10k steps: target updated every 10k learning steps
- Longer → more stable, but target drifts further
- Shorter → follows current θ faster, but noisier

### Loss Function
- MSE (mean squared error) or Huber loss (robust to outliers)
- Huber clips large errors: reduces impact of noisy Q-estimates

## Extensions and Variants

### Double DQN
Problem: max_a Q(s',a;θ⁻) is biased (overestimates)
Solution: Use current network for action selection, target network for value:
$$y = r + \gamma Q(s', \text{argmax}_a Q(s',a;\theta); \theta^-)$$

### Dueling DQN
Problem: Q(s,a) conflates state value and action advantage
Solution: Separate: $Q(s,a) = V(s) + A(s,a) - \bar{A}(s)$
- Value head: how good is this state?
- Advantage head: how much better is this action?

### Prioritized Experience Replay (PER)
Problem: Uniform sampling treats all transitions equally
Solution: Sample by TD error priority (more surprising = more important)
- Surprising transitions are likely to provide more learning signal

### Rainbow
Combines all advances: Double + Dueling + PER + n-step returns + distributional Q + noisy nets

## Why DQN Works

### Breaking the Deadly Triad
- Deadly triad: approximation + bootstrapping + off-policy → divergence
- DQN reduces one factor:
  - Experience replay: weakens temporal correlation (mitigates off-policy issues)
  - Fixed targets: breaks tight coupling between current and target values

### Convergence Properties
Not guaranteed to converge to optimal, but empirically works well:
- Replay + fixed targets stabilize learning enough for SGD to work
- Neural network capacity approximates Q well enough
- Practical trade-off: stability over guarantees

## Summary

### Key Innovations
1. **Experience Replay**: Store and resample old transitions
2. **Fixed Q-Targets**: Separate network for stable targets
3. **ε-Decay**: Explore early, exploit later

### Impact
- First scalable deep RL algorithm (Atari results, 2013)
- Foundation for modern deep RL
- Variants (Double, Dueling, PER) still competitive today

### Next: Lesson 7B (Practical)
Implement DQN from scratch in PyTorch on CartPole → Atari games